# D'QUILANE — Train + Web Tester (All-in-One)

Notebook ini **training + langsung buka web tester** buat D'QUILANE.

## Cara pakai di Kaggle:
1. Settings (kanan) → Accelerator → **GPU T4 x2**
2. Settings → Internet → **On**
3. Run All
4. Setelah cell terakhir jalan, akan muncul **public URL Gradio** di output
5. Klik URL itu → web tester D'QUILANE siap dipakai di browser!

In [ ]:
# ===== CELL 1: INSTALL DEPS =====
import subprocess, sys
def install(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', *pkgs, '-q'])

for pkg in ['torch', 'gradio']:
    try:
        __import__(pkg)
    except ImportError:
        install(pkg)

import torch
import gradio as gr
print(f'PyTorch: {torch.__version__}')
print(f'Gradio:  {gr.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nUsing device: {DEVICE}')

In [ ]:
# ===== CELL 2: CONFIG =====
CONFIG = {
    'vocab_size': 256,   # akan di-override setelah tokenizer
    'd_model':    128,
    'n_heads':    4,
    'd_ff':       512,
    'n_layers':   4,    # virtual depth (parameter sharing)
    'max_seq':    64,
    'epochs':     1000,
    'batch_size': 16,
    'lr':         3e-4,
}

# ===== DATASET: tambahin teks Indonesia di sini =====
TEXT = '''Indonesia adalah negara kepulauan.
Ibu kota Indonesia adalah Jakarta.
Bahasa Indonesia adalah bahasa resmi.
Mata uang Indonesia adalah rupiah.
Indonesia kaya akan sumber daya alam.
Penduduk Indonesia sangat beragam.
Setiap suku memiliki budaya yang unik.
Gunung berapi banyak di Indonesia.
Hutan tropis Indonesia terbesar di dunia.
Laut Indonesia kaya akan ikan.
Iklim Indonesia adalah tropis.
Musim hujan dan kemarau silih berganti.
Pertanian adalah mata pencaharian utama.
Padi ditanam di sawah oleh petani.
Nasi adalah makanan pokok Indonesia.
Pendidikan kunci masa depan bangsa.
Teknologi berkembang pesat di era modern.
Komputer dan internet ubah cara komunikasi.
Kecerdasan buatan menjadi tren global.
Model bahasa besar sedang populer sekarang.
Transformer arsitektur dasar model modern.
Attention mekanisme memahami konteks kalimat.
Parameter sharing hemat ukuran model.
ALBERT contoh model dengan parameter sharing.
Belajar setiap hari bikin hidup bermakna.
Kerja keras kunci pencapaian tujuan.
Indonesia akan terus maju dan berkembang.
Anak muda harapan masa depan bangsa.
Mari bangun negeri dengan semangat pantang menyerah.
Pahlawan nasional Indonesia banyak jumlahnya.
Proklamasi kemerdekaan dibacakan tujuh belas agustus.
Pendidikan penting untuk masa depan anak.
Indonesia merdeka tahun seribu sembilan ratus empat puluh lima.
Bendera Indonesia merah putih.
Lagu kebangsaan Indonesia Raya.
Pulau Jawa adalah pulau terpadat di Indonesia.
Sumatra Kalimantan Sulawesi Papua pulau besar lainnya.
Bali pulau terkenal dengan pariwisata.
Danau Toba di Sumatra Utara.
Raja Ampat di Papua barat daya.
Borobudur candi Buddha terbesar di dunia.
Prambanan candi Hindu terbesar di Indonesia.
Komodo hewan purba di Nusa Tenggara Timur.
Orangutan hewan endemik Sumatra dan Kalimantan.
Badak Jawa hewan langka di Ujung Kulon.
Bunga Rafflesia bunga terbesar di dunia.
Indonesia kaya budaya dan alam.
'''

In [ ]:
# ===== CELL 3: TOKENIZER (char-level, simpel) =====
chars = sorted(set(TEXT))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
stoi['<pad>'] = len(stoi)
itos[len(itos)] = '<pad>'
PAD_ID = stoi['<pad>']
VOCAB = len(stoi)
CONFIG['vocab_size'] = VOCAB

def encode(s):
    return [stoi[c] for c in s if c in stoi]

def decode(ids):
    return ''.join(itos.get(int(i), '?') for i in ids if i != PAD_ID)

data = torch.tensor(encode(TEXT), dtype=torch.long)
print(f'Vocab size: {VOCAB} karakter')
print(f'Total tokens: {len(data)}')

def get_batch():
    idx = torch.randint(0, len(data) - CONFIG['max_seq'] - 1, (CONFIG['batch_size'],))
    x = torch.stack([data[i:i+CONFIG['max_seq']] for i in idx])
    y = torch.stack([data[i+1:i+1+CONFIG['max_seq']] for i in idx])
    return x.to(DEVICE), y.to(DEVICE)

In [ ]:
# ===== CELL 4: MODEL D'QUILANE =====
import torch.nn as nn
import torch.nn.functional as F
import math

class Attention(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        assert d % h == 0
        self.h, self.dh = h, d // h
        self.qkv = nn.Linear(d, 3*d, bias=False)
        self.out = nn.Linear(d, d, bias=False)
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.h, self.dh)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.dh)
        mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.out(out)

class FFN(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.gate = nn.Linear(d, d_ff, bias=False)
        self.val  = nn.Linear(d, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d, bias=False)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.val(x))

class Block(nn.Module):
    def __init__(self, d, h, d_ff):
        super().__init__()
        self.norm1 = nn.RMSNorm(d)
        self.attn  = Attention(d, h)
        self.norm2 = nn.RMSNorm(d)
        self.ffn   = FFN(d, d_ff)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class DQuilane(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg['vocab_size'], cfg['d_model'])
        self.pos_emb = nn.Embedding(cfg['max_seq'], cfg['d_model'])
        self.block = Block(cfg['d_model'], cfg['n_heads'], cfg['d_ff'])
        self.norm_f = nn.RMSNorm(cfg['d_model'])
        self.head = nn.Linear(cfg['d_model'], cfg['vocab_size'], bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        self.apply(self._init)
    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0, std=0.02)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        for _ in range(self.cfg['n_layers']):
            x = self.block(x)
        x = self.norm_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.cfg['vocab_size']),
                                    targets.view(-1))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new=50, temp=0.8, top_k=10):
        self.eval()
        for _ in range(max_new):
            x = idx if idx.size(1) <= self.cfg['max_seq'] else idx[:, -self.cfg['max_seq']:]
            logits, _ = self(x)
            logits = logits[:, -1, :] / temp
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, 1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

model = DQuilane(CONFIG).to(DEVICE)
n = sum(p.numel() for p in model.parameters())
print(f"D'QUILANE lahir! Unique params: {n:,} (~{n*4/1024:.1f} KB)")
print(f'Virtual depth: {CONFIG["n_layers"]} (effective: {n * CONFIG["n_layers"]:,})')

In [ ]:
# ===== CELL 5: TRAINING =====
import time
opt = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=0.01)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CONFIG['epochs'])

model.train()
t0 = time.time()
losses = []
for epoch in range(CONFIG['epochs']):
    xb, yb = get_batch()
    opt.zero_grad()
    _, loss = model(xb, yb)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    sched.step()
    losses.append(loss.item())
    if epoch % 100 == 0 or epoch == CONFIG['epochs'] - 1:
        print(f'epoch {epoch+1:4d}/{CONFIG["epochs"]}  loss={loss.item():.4f}  t={time.time()-t0:.1f}s')

print(f'\nTraining done in {time.time()-t0:.1f}s')
print(f'Loss: {losses[0]:.4f} -> {losses[-1]:.4f}')
model.eval()

In [ ]:
# ===== CELL 6: SAVE MODEL =====
import os
os.makedirs('output', exist_ok=True)
torch.save({
    'state_dict': model.state_dict(),
    'config': CONFIG,
    'stoi': stoi,
    'itos': {int(k): v for k, v in itos.items()},
}, 'output/dquilane_balita.pt')
size = os.path.getsize('output/dquilane_balita.pt') / 1024
print(f'Model saved: output/dquilane_balita.pt ({size:.1f} KB)')
print('Download di tab Output (kanan)')

In [ ]:
# ===== CELL 7: WEB TESTER PAKAI GRADIO =====
# Bikin web UI langsung di output notebook. Public URL bisa dishare.

def generate_text(prompt, max_new=100, temp=0.7, top_k=10):
    if not prompt:
        return '[masukkan prompt dulu]'
    ids = encode(prompt)
    if not ids:
        return '[prompt gak ada karakter yang dikenal]'
    idx = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    out = model.generate(idx, max_new=int(max_new), temp=float(temp), top_k=int(top_k))
    return decode(out[0].tolist())

def generate_streaming(prompt, max_new, temp, top_k):
    """Streaming generate — biar terlihat token by token."""
    if not prompt:
        yield '[masukkan prompt dulu]'
        return
    ids = encode(prompt)
    if not ids:
        yield '[prompt gak ada karakter yang dikenal]'
        return
    idx = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    accumulated = prompt
    yield accumulated
    for _ in range(int(max_new)):
        x = idx if idx.size(1) <= CONFIG['max_seq'] else idx[:, -CONFIG['max_seq']:]
        logits, _ = model(x)
        logits = logits[:, -1, :] / float(temp)
        if int(top_k) > 0:
            v, _ = torch.topk(logits, min(int(top_k), logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, 1)
        idx = torch.cat([idx, next_id], dim=1)
        accumulated += itos.get(int(next_id.item()), '?')
        yield accumulated

# ===== Bikin UI =====
with gr.Blocks(title="D'QUILANE Web Tester") as demo:
    gr.Markdown("""
    # D'QUILANE Web Tester v0.2
    LLM buatan sendiri dari nol. Trained in this notebook, **tidak quantize**.
    
    **Teknik**: Transformer + Parameter Sharing (ALBERT-style).
    **Skill level**: Balita (baru belajar ngomong 😅).
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(
                label='Prompt',
                placeholder='Ketik apa aja, mis: Indonesia, Belajar, Teknologi...',
                value='Indonesia',
                lines=2,
            )
            max_new = gr.Slider(20, 300, value=100, step=10, label='Max tokens')
            temp = gr.Slider(0.1, 2.0, value=0.7, step=0.1, label='Temperature (kreativitas)')
            top_k = gr.Slider(0, 50, value=10, step=1, label='Top-k (0 = off)')
            btn = gr.Button('Generate', variant='primary')
            clear = gr.Button('Clear')
        
        with gr.Column(scale=1):
            output = gr.Textbox(label='Output', lines=15, show_copy_button=True)
    
    # event handlers
    btn.click(generate_streaming, [prompt, max_new, temp, top_k], output)
    clear.click(lambda: '', None, output)
    
    gr.Examples(
        examples=[
            ['Indonesia'],
            ['Belajar'],
            ['Padi'],
            ['Teknologi'],
            ['Mari bangun'],
            ['Kerja keras'],
        ],
        inputs=prompt,
    )
    
    gr.Markdown("""
    ---
    
    **Tips**:
    - Temperature rendah (0.3) = output konsisten tiap run
    - Temperature tinggi (1.2+) = output kreatif tapi random
    - Top-k = 3 = fokus ke 3 token teratas (output monoton)
    - Top-k = 0 = mati, semua vocab dipertimbangkan
    
    **Warning**: Model ini masih level balita (loss ~1.0). Output gak akan masuk akal sempurna. Latih lebih lama + dataset lebih besar untuk naik level.
    """)

# ===== LAUNCH =====
# Di Kaggle, share=True akan bikin public URL yang bisa diakses dari mana aja
# URL aktif selama notebook jalan. Kalau notebook di-stop, URL mati.
demo.launch(share=True, debug=False)
# Kalau mau launch tanpa share (cuma bisa diakses di tab output Kaggle):
# demo.launch(share=False)

## 🎉 Setelah Run All selesai:

1. Cell 7 akan nge-print public URL seperti:
   ```
   Running on local URL:  http://127.0.0.1:7860
   Running on public URL: https://xxxx.gradio.live
   ```
2. **Klik public URL itu** → buka tab baru di browser
3. Web tester D'QUILANE siap dipakai!

## ⚠️ Catatan penting:

- Public URL aktif **selama notebook masih running**. Kalau di-stop, URL mati.
- URL otomatis expired dalam 72 jam (limit Gradio free).
- Kalau mau URL persistent, hosting model ke **Hugging Face Spaces** (bisa aku bantu nanti).

## 💡 Edit & eksperimen:

- Ganti `TEXT` di cell 2 dengan teks 10x lebih banyak → output lebih koheren
- Naikkan `epochs` ke 5000 → loss lebih rendah
- Naikkan `d_model` 128 → 256 → output lebih kompleks (file size naik 4x)